# Notebook 4 — Kết quả Thực nghiệm: Phân tích và Thảo luận

**Bài tập lớn số 2 · CO5085 · HCMUT 2025-2026**

## Nội dung
1. Bảng số liệu tổng hợp
2. Biểu đồ so sánh mAP
3. Biểu đồ Speed vs Accuracy
4. Per-class AP analysis
5. Visualize kết quả định tính (qualitative)
6. Phân tích lỗi (TP/FP/FN)
7. Thảo luận và kết luận

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.utils import (load_metrics_json, print_detection_results_table,
                        plot_map_comparison, plot_speed_accuracy_tradeoff,
                        plot_per_class_ap, visualize_detections)
from src.data import VOCDetectionDataset, get_val_transforms, get_device, VOC_CLASSES

device = get_device()
print("Device:", device)

## 1. Bảng số liệu tổng hợp

In [ ]:
try:
    yolo_r = load_metrics_json('../results/metrics/yolo_results.json')
    frcnn_r = load_metrics_json('../results/metrics/frcnn_results.json')
    results = {
        yolo_r.get('model', 'YOLOv8n'): yolo_r,
        frcnn_r.get('model', 'Faster R-CNN'): frcnn_r,
    }
    print_detection_results_table(results)
    has_results = True
except FileNotFoundError:
    print("[WARN] Chưa có kết quả. Chạy training scripts trước.")
    has_results = False

## 2. Biểu đồ so sánh mAP

In [ ]:
if has_results:
    plot_map_comparison(results, save_path='../results/plots/map_comparison.png')
else:
    print("Cần kết quả training trước khi vẽ biểu đồ.")

## 3. Speed vs Accuracy Trade-off

In [ ]:
if has_results:
    plot_speed_accuracy_tradeoff(results, save_path='../results/plots/speed_accuracy.png')
    print()
    print("Nhận xét quan trọng:")
    print("→ One-stage (YOLO): nhanh hơn nhiều, phù hợp real-time (camera, video)")
    print("→ Two-stage (Faster R-CNN): chính xác hơn, phù hợp khi không cần real-time")

## 4. Per-Class AP

In [ ]:
if has_results and 'AP_per_class' in frcnn_r:
    plot_per_class_ap(results, save_path='../results/plots/per_class_ap.png')

    # Top 5 dễ nhất và khó nhất
    ap_frcnn = frcnn_r.get('AP_per_class', {})
    if ap_frcnn:
        sorted_ap = sorted(ap_frcnn.items(), key=lambda x: x[1], reverse=True)
        print("\nTop 5 lớp dễ nhất (Faster R-CNN AP@0.5):")
        for cls, ap in sorted_ap[:5]:
            print(f"  {cls:15s}: {ap*100:.1f}%")
        print("\nTop 5 lớp khó nhất:")
        for cls, ap in sorted_ap[-5:]:
            print(f"  {cls:15s}: {ap*100:.1f}%")
else:
    print("Cần per-class AP trong results JSON.")

## 5. Visualize kết quả định tính

In [ ]:
# Load model và visualize predictions
try:
    frcnn_path = '../results/checkpoints/frcnn_voc.pth'
    from src.models import get_faster_rcnn
    from src.train import load_frcnn_checkpoint

    model = get_faster_rcnn(num_classes=21)
    model = load_frcnn_checkpoint(model, frcnn_path, device)
    model.eval()

    ds = VOCDetectionDataset('../data/voc', year='2012', image_set='val',
                              transforms=get_val_transforms())

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    import torch
    with torch.no_grad():
        for i, ax in enumerate(axes):
            img_t, target = ds[i * 50]
            output = model([img_t.to(device)])[0]

            keep = output['scores'] > 0.5
            visualize_detections(
                image=img_t,
                boxes=output['boxes'][keep].cpu().tolist(),
                labels=output['labels'][keep].cpu().tolist(),
                scores=output['scores'][keep].cpu().tolist(),
                gt_boxes=target['boxes'].tolist(),
                gt_labels=target['labels'].tolist(),
                title=f"Val #{i*50} — GT(đỏ/nét đứt) vs Pred(màu)",
                ax=ax,
            )

    plt.suptitle('Faster R-CNN Predictions trên Pascal VOC val', fontsize=12)
    plt.tight_layout()
    plt.savefig('../results/plots/qualitative_frcnn.png', dpi=100)
    plt.show()
except Exception as e:
    print(f"[WARN] Không thể visualize: {e}")
    print("Cần có checkpoint frcnn_voc.pth")

## 6. Phân tích lỗi (Error Analysis)

In [ ]:
try:
    from src.evaluate import predict_frcnn, get_detection_errors
    from src.data import get_frcnn_loaders

    model = get_faster_rcnn(num_classes=21)
    model = load_frcnn_checkpoint(model, '../results/checkpoints/frcnn_voc.pth', device)

    _, val_loader = get_frcnn_loaders('../data/voc', batch_size=4)
    preds, targets = predict_frcnn(model, val_loader, device)
    errors = get_detection_errors(preds, targets, iou_threshold=0.5, score_threshold=0.5)

    print("=== Error Analysis (Faster R-CNN) ===")
    print(f"  TP: {errors['TP']}")
    print(f"  FP: {errors['FP']}  (phát hiện sai)")
    print(f"  FN: {errors['FN']}  (bỏ sót)")
    print(f"  Precision: {errors['precision']*100:.1f}%")
    print(f"  Recall:    {errors['recall']*100:.1f}%")
    print(f"  F1:        {errors['f1']*100:.1f}%")

    # FN cao nhất (bỏ sót nhiều nhất)
    fn_sorted = sorted(errors['fn_per_class'].items(), key=lambda x: -x[1])[:5]
    print("\nTop 5 lớp bị bỏ sót nhiều nhất (FN):")
    for cls, cnt in fn_sorted:
        if cnt > 0:
            print(f"  {cls}: {cnt}")
except Exception as e:
    print(f"Cần có checkpoint: {e}")

## 7. Thảo luận và Kết luận

### Kết quả

*(Điền số liệu thực sau khi chạy training)*

| Mô hình | mAP@0.5 | mAP@0.5:0.95 | FPS (CPU) | Params |
|---------|---------|--------------|-----------|--------|
| YOLOv8n | TBD | TBD | TBD | 3.2M |
| Faster R-CNN ResNet-50 FPN | TBD | TBD | TBD | 41.8M |

### Phân tích

**Độ chính xác:**
- Faster R-CNN có xu hướng chính xác hơn vì two-stage pipeline cho phép tập trung vào từng region
- YOLOv8 có thể kém hơn ở small objects (objects nhỏ) nhưng tốt hơn ở single dominant objects

**Tốc độ:**
- YOLOv8n nhanh hơn Faster R-CNN ~5-10x
- Single forward pass vs RPN + ROI head

**Khi nào dùng mô hình nào?**
- YOLOv8: ứng dụng real-time (camera, video, autonomous driving)
- Faster R-CNN: cần độ chính xác cao, không yêu cầu real-time (medical imaging, surveillance)

### Hạn chế
- Training trên GPU sẽ cho kết quả tốt hơn nhiều (cần ít nhất 8GB VRAM)
- VOC 2012 chỉ có 20 lớp — kết quả trên COCO (80 lớp) sẽ khác